# Activity

Out choice of 10 for K was arbitrary - what effect do differejt K values on the results?

Our distance metric is also somewhat arbitrary - we just took the cosine distance between the genres and added it to the ifference between the normalized popularity scores, CAn u improve on that?

In [1]:
import numpy as np
import pandas as pd

In [3]:
r_cols = ['user_id', 'movie_id', 'rating'] #define as colunas
ratings = pd.read_csv('../anexos/Aquivos_de_C%C3%B3digo/u.data', sep='\t', names=r_cols, usecols=range(3)) #le o csv e coloca as colunas
ratings.head()

,user_id,movie_id,rating
0,0,50,5
1,0,172,5
2,0,133,1
3,196,242,3
4,186,302,3


In [5]:
movieProperties = ratings.groupby('movie_id').agg({'rating': [np.size, np.mean]}) #agrupa os dados por id e faz o tanto de avaliacoes e a media delas

/tmp/ipykernel_7264/1750236370.py:1: FutureWarning: The provided callable <function mean at 0x7f3eb026d590> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  movieProperties = ratings.groupby('movie_id').agg({'rating': [np.size, np.mean]}) #agrupa os dados por id e faz o tanto de avaliacoes e a media delas


In [6]:
movieNumRatings = pd.DataFrame(movieProperties['rating']['size']) #faz um dataframe acessando o rating e o size do movieproperties
movieNormalizedNumRatings = movieNumRatings.apply(lambda x: (x - np.min(x)) / (np.max(x) - np.min(x))) #normaliza os numeros 

In [8]:
movieDict = {}
with open('../anexos/Aquivos_de_C%C3%B3digo/u.item', encoding='latin-1') as f: #arquivo com os dados
    temp = ''
    for line in f: #pra cada linha dentro do arquivo
        fields = line.rstrip('\n').split('|') #separa com | 
        movieID = int(fields[0]) #pega o id do filme e coloca como int
        name = fields[1] #pega o nome do filme
        genres = fields[5:25] #pega os generos do 5 ao 25
        genres = map(int, genres) #converte pra int
        movieDict[movieID] = (name, genres, movieNormalizedNumRatings.loc[movieID].get('size'),movieProperties.loc[movieID].rating.get('mean'))
        #joga pro dicionario o filme

In [9]:
from scipy import spatial

In [10]:
def ComputeDistance(a, b): #funcao que pega a diferenca dos generos de filme e da popularidade e ve se eles sao mt diferentes 
    genreA = list(a[1])
    genreB = list(b[1])

    if len(genreA) == 0 or len(genreB) == 0:
        return 1

    genreDistance = spatial.distance.cosine(genreA, genreB) #calcula a distancia do cosseno 

    popularityA = a[2]
    popularityB = b[2]

    popularityDistance = abs(popularityA - popularityB) #abs e pra deixar sempre positivo e aq pega a distancia da popularidade dos dois

    return genreDistance + popularityDistance 


ComputeDistance(movieDict[2], movieDict[4])

np.float64(0.8004574042309892)

In [15]:
import operator

def getNeighbors(movieID, K):
    distances = []
    for movie in movieDict: #cada filme no dicionario
        if (movie != movieID): #nao pega o mesmo filme
            dist = ComputeDistance(movieDict[movieID], movieDict[movie]) #pega a distancia entre eles
            distances.append((movie, dist)) #adiciona o filme e a distancia na lista

    distances.sort(key = operator.itemgetter(1)) #ordena pela menor distancia
    neighbors = []
    for x in range(K): #repete K vezes
        neighbors.append(distances[x][0]) #na posicao x da lista pega o id do filme

    return neighbors


K = 3
avgRating = 0
neighbors = getNeighbors(1, K)

for neighbor in neighbors:
    avgRating += movieDict[neighbor][3] #pega a nota do filme e joga pro avgrating
    print (movieDict[neighbor][0] + " " + str(movieDict[neighbor][3])) 

avgRating /= float(K) #pega o avrating e divide pela qntd de K

GoldenEye (1995) 3.2061068702290076
Four Rooms (1995) 3.033333333333333
Get Shorty (1995) 3.550239234449761


In [16]:
print(avgRating) #media pequena

3.263226479337367


In [17]:
K = 50
avgRating = 0
neighbors = getNeighbors(1, K)

for neighbor in neighbors:
    avgRating += movieDict[neighbor][3] #pega a nota do filme e joga pro avgrating
    print (movieDict[neighbor][0] + " " + str(movieDict[neighbor][3])) 

avgRating /= float(K) #pega o avrating e divide pela qntd de K

GoldenEye (1995) 3.2061068702290076
Four Rooms (1995) 3.033333333333333
Get Shorty (1995) 3.550239234449761
Copycat (1995) 3.302325581395349
Shanghai Triad (Yao a yao yao dao waipo qiao) (1995) 3.576923076923077
Twelve Monkeys (1995) 3.798469387755102
Babe (1995) 3.9954337899543377
Dead Man Walking (1995) 3.8963210702341136
Richard III (1995) 3.831460674157303
Seven (Se7en) (1995) 3.847457627118644
Usual Suspects, The (1995) 4.385767790262173
Mighty Aphrodite (1995) 3.4184782608695654
Postino, Il (1994) 3.9672131147540983
Mr. Holland's Opus (1995) 3.7781569965870307
French Twist (Gazon maudit) (1995) 3.2051282051282053
From Dusk Till Dawn (1996) 3.119565217391304
White Balloon, The (1995) 2.8
Antonia's Line (1995) 3.9565217391304346
Angels and Insects (1995) 3.4166666666666665
Muppet Treasure Island (1996) 2.761904761904762
Braveheart (1995) 4.151515151515151
Taxi Driver (1976) 4.1208791208791204
Rumble in the Bronx (1995) 3.4482758620689653
Birdcage, The (1996) 3.4436860068259385
Brot

In [18]:
print(avgRating) #aumento na media

3.4423851983058693


In [19]:
K = 200
avgRating = 0
neighbors = getNeighbors(1, K)

for neighbor in neighbors:
    avgRating += movieDict[neighbor][3] #pega a nota do filme e joga pro avgrating
    print (movieDict[neighbor][0] + " " + str(movieDict[neighbor][3])) 

avgRating /= float(K) #pega o avrating e divide pela qntd de K

GoldenEye (1995) 3.2061068702290076
Four Rooms (1995) 3.033333333333333
Get Shorty (1995) 3.550239234449761
Copycat (1995) 3.302325581395349
Shanghai Triad (Yao a yao yao dao waipo qiao) (1995) 3.576923076923077
Twelve Monkeys (1995) 3.798469387755102
Babe (1995) 3.9954337899543377
Dead Man Walking (1995) 3.8963210702341136
Richard III (1995) 3.831460674157303
Seven (Se7en) (1995) 3.847457627118644
Usual Suspects, The (1995) 4.385767790262173
Mighty Aphrodite (1995) 3.4184782608695654
Postino, Il (1994) 3.9672131147540983
Mr. Holland's Opus (1995) 3.7781569965870307
French Twist (Gazon maudit) (1995) 3.2051282051282053
From Dusk Till Dawn (1996) 3.119565217391304
White Balloon, The (1995) 2.8
Antonia's Line (1995) 3.9565217391304346
Angels and Insects (1995) 3.4166666666666665
Muppet Treasure Island (1996) 2.761904761904762
Braveheart (1995) 4.151515151515151
Taxi Driver (1976) 4.1208791208791204
Rumble in the Bronx (1995) 3.4482758620689653
Birdcage, The (1996) 3.4436860068259385
Brot

In [20]:
print(avgRating)

3.553786164258346
